In [2]:
from langgraph.graph import StateGraph, END, START
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage

In [3]:
genrator_llm = ChatGroq(model="openai/gpt-oss-120b")

evaluator_llm = ChatGroq(model="openai/gpt-oss-120b")

optimizer_llm = ChatGroq(model="openai/gpt-oss-120b")

In [4]:
class TweetState(TypedDict):

    topic: str
    tweet: str
    evalution: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

In [6]:
def generate(state: TweetState):

    message = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""Write a short, original, and hilarious tweet on the topic: "{state['topic']}".
            Rules:
            - Do NOT use question-answer format.
            - Max 280 characters.
            - Use observational humor, irony, sarcasm, or cultural references.
            - Think in meme logic, punchlines, or relatable takes.
            - Use simple, day to day english.
        """)
    ]

    output = genrator_llm.invoke(message).content

    return {"tweet": output}

In [7]:
class TweetEvaluation (BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="Constructive feedback for the tweet.")

structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [8]:
def evaluate(state: TweetState):

    messages = [
        SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
        HumanMessage(content=f"""
            Evaluate the following tweet:

            Tweet: "{state['tweet']}"

            Use the criteria below to evaluate the tweet:

            1. Originality - Is this fresh, or have you seen it a hundred times before?
            2. Humor - Did it genuinely make you smile, laugh, or chuckle?
            3. Punchiness - Is it short, sharp, and scroll-stopping?
            4. Virality Potential - Would people retweet or share it?
            5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

            Auto-reject if:
            - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
            - It exceeds 280 characters
            - It reads like a traditional setup-punchline joke
            - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-uncle universe" or vague summaries)

            ### Respond ONLY in structured format:
            evaluation: "approved" or "needs_improvement"
            feedback: One paragraph explaining the strengths and weaknesses
        """)
    ]

    response = structured_evaluator_llm.invoke(messages)

    return {"evaluation": response.evalution, "feedback": response.feedback}

In [9]:
def optimize(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
            Improve the tweet based on this feedback:
            "{state['feedback']}"

            Topic: "{state['topic']}"
            Original Tweet:
            {state['tweet']}

            Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
            """)
        ]

    response = optimizer_llm.invoke(messages).content

    iteration = state["iteration"]+1

    return {"tweet": response, "iteration": iteration}

In [ ]:
graph = StateGraph(TweetState)

graph.add_node("generate", generate)
graph.add_node("evaluate", evaluate)
graph.add_node("optimize", optimize)

graph.add_edge()